# Covariate Data Formatting



## Description

Our covariate preprocessing steps merge genotypic principal components and fixed covariate files into one file for downstream QTL analysis. 

## Input Files

| File | Description |
|------|-------------|
| `protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.rds` | Genotype PCA result (RDS) produced by the genotype PCA module; contains the `pc_scores` matrix of per-sample principal components. |
| `protocol_example.covariates.tsv` | Fixed/known covariates (e.g. sex, age, PMI, study), samples in rows and covariates in columns with an `#id` header. |

## Output Files

| File | Description |
|------|-------------|
| `protocol_example.covariates.protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.gz` | Combined covariate matrix: the fixed covariates stacked with the selected genotype PCs, samples as columns and an `#id` header, ready for downstream QTL analysis. |

## Minimal Working Example Steps

The data and singularity used in this minimal working example can be found on [Synapse](https://www.synapse.org/#!Synapse:syn69670658).

### Merge Covariates and Genotype PCs

The data and singularity used in this minimal working example can be found on [Synapse](https://www.synapse.org/#!Synapse:syn69670658).

Timing: <1 min

This step reads the genotype PCA result (`--pcaFile`) and the fixed covariate table (`--covFile`), keeps the samples present in both, and stacks the chosen genotype PCs on top of the covariates to produce a single matrix for QTL analysis. The number of PCs to keep is set with `--k`; here we pick the count whose cumulative variance explained stays under 80% by reading the PCA scree file. `--tol-cov 0.4` allows samples with up to 40% missing covariate values to be mean-imputed rather than dropped.

In [ ]:
sos run pipeline/covariate_formatting.ipynb merge_genotype_pc \
    --cwd output/covariate/ \
    --pcaFile output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.rds \
    --covFile input/covariate/protocol_example.covariates.tsv \
    --tol-cov 0.4 \
    --k `awk '$3 < 0.8' output/genotype/genotype_pca/protocol_example.genotype.merged.plink_qc.plink_qc.prune.pca.scree.txt | tail -1 | cut -f 1`

## Command Interface

In [ ]:
sos run covariate_formatting.ipynb -h

## Setup and global parameters

In [ ]:
[global]
# The output directory for generated files. 
parameter: cwd = path("output")
# The covariate file
parameter: covFile = path
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "2G"
# Number of threads
parameter: numThreads = 8
# Software container option
parameter: container = ""
cwd = path(f"{cwd:a}")

### **Step 0.** Merge Covariates and Genotype PCs

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[merge_genotype_pc]
# An RDS file as the output of the genotype PCA module
parameter: pcaFile = path
# The number of PCs to retain, by default is 20, in practice can be the number of PC that captured more than 70% PVE
parameter: k = 20
parameter: name = f'{covFile:bn}.{pcaFile:bn}'
# Outliers
parameter: outliersFile = path(".") 
parameter: remove_outliers = False
# Tolerance of missingness in covariates, -1 means do nothing, otherwise for samples with covariates missing rate larger than tol_cov will be removed,
# with missing rate smaller than tol_cov will be kept.
parameter: tol_cov = -1.0 
parameter: mean_impute = True
stop_if(remove_outliers and not outliersFile.is_file(), msg = "No outliers file specified, please add outliers file or remove the remove-outliers flag")
input: pcaFile, covFile
output:  f'{cwd:a}/{name}.gz'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand= "$[ ]", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
        library("dplyr")
        library("readr")
        library("data.table")

        compute_missing <- function(mtx){
          miss <- sum(is.na(mtx))/length(mtx)
          return(miss)
        }

        mean_impute <- function(mtx){
          f <- apply(mtx, 2, function(x) mean(x,na.rm = TRUE))
          for (i in 1:length(f)) mtx[,i][which(is.na(mtx[,i]))] <- f[i]
          return(mtx)
        }

        filter_mtx <- function(X, missing_rate_thresh) {
            rm_col <- which(apply(X, 2, compute_missing) > missing_rate_thresh)
            if (length(rm_col)) X <- X[, -rm_col]
            return($['mean_impute(X)' if mean_impute else 'X'])
        }  

        if ($["TRUE" if _input[0].suffix.lower() == '.rds' else "FALSE" ]) {
          pca_output = readRDS("$[_input[0]]")$pc_scores
        } else {
          pca_putput = fread("$[_input[0]]")
        }
        
        mtx = pca_output%>%select(contains("PC"))%>%t()
        colnames(mtx) <- pca_output$IID
        ## Keep only the number of PCs specified
        mtx = mtx[1:$[k],]
        mtx = mtx%>%as_tibble(rownames = "#id")
        ## Load covariates
        covt = transpose(fread("$[_input[1]]", head=T), keep.names="#id", make.names=1) %>% as_tibble()

        ## Retaining only the overlapped samples with genotype
        overlap = intersect(colnames(covt),colnames(mtx))
        
        ## Report sample counts:
        print(paste(ncol(covt) - 1, "samples are in the covariate file", sep = " "))
        print(paste(ncol(mtx), "samples are in the PCA file", sep = " "))

        ## Report identical samples:
        print(paste(length(overlap) - 1, "samples overlap between covariate & PCA files and are included in the analysis:", sep = " "))
        print(overlap[!overlap == '#id'])

        ## Report non-overlapping samples:
        cov_missing = covt %>% select(-all_of(overlap))
        print(paste(ncol(cov_missing), "samples in the covariate file are missing from the PCA file:", sep = " "))
        print(colnames(cov_missing))

        pca_missing = mtx %>% select(-all_of(overlap))
        print(paste(ncol(pca_missing), "samples in the PCA file are missing from the covariate file:", sep = " "))
        print(colnames(pca_missing))
        
        ## Removal of outlier if needed
        if ($["TRUE" if remove_outliers else "FALSE"]){
              outlier = fread("$[outliersFile]", head = FALSE)$V2 %>% as_tibble()
              overlap = setdiff(overlap,outlier)
              }
        covt = covt%>%select(all_of(overlap))

        mtx = mtx%>%select(all_of(overlap))
        output = bind_rows(covt,mtx)

        ## Handle missingess in covariates
        if($[tol_cov] == -1){if(sum(is.na(output)) > 0 ){ stop("NA in covariates input: Check input file or set parameter tol_cov to allow for removing missing values; mean_impute to allow for imputing missing values")}}
        output = output%>%as.data.frame
        rownames(output) = output$`#id`
        output = filter_mtx(output[,2:ncol(output)],$[tol_cov])%>%as_tibble(rownames = "#id")
        output%>%write_delim("$[_output]","\t")

bash: expand= "$[ ]", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
        stdout=$[_output:n].stdout 
        for i in $[_output] ; do 
        echo "output_info: $i " >> $stdout;
        echo "output_size:" `ls -lh $i | cut -f 5  -d  " "`   >> $stdout;
        echo "output_rows:" `zcat $i | wc -l  | cut -f 1 -d " "`   >> $stdout;
        echo "output_column:" `zcat $i | head -1 | wc -w `   >> $stdout;
        echo "output_preview:"   >> $stdout;
        zcat $i | head  | cut -f 1,2,3,4,5,6   >> $stdout ; done
